# Scoreboard

> Every forecasting model on this site, scored on the same backtest: one table per dataset, best first.

- skip_showdoc: true

## How to read it

Each forecasting page calls `record` from `ML_Timeseries.backtest`, which writes one JSON file per model into `nbs/results/`. This page reads them all. A page re-run updates its own row; nothing else changes.

- **MASE**: mean absolute scaled error, the headline. Lower is better; 1.0 is as good as seasonal naive was on the history.
- **Skill**: the share of seasonal naive's error removed on the same test windows. Higher is better; 0 is the reference itself.
- **Seconds**: total forecasting time across the 30 windows, including any refits, on a 4-core CPU or a 12 GB GPU.

The [Backtesting](00_Backtesting.ipynb) page defines the household protocol and [Benchmarks](02_Benchmarks.ipynb) the M4 Hourly one. [Forecast Metrics](01_Forecast_Metrics.ipynb) explains why these metrics.

---

In [ ]:
from ML_Timeseries.backtest import load_results
from ML_Timeseries.data import bar_chart, show

In [ ]:
res = load_results()
cols = ["model", "family", "mase", "skill", "mae", "wape", "seconds", "notebook"]
print(f"{len(res)} rows from {res.notebook.nunique()} notebooks")

41 rows from 22 notebooks


## Household energy, day-ahead

One house, hourly kWh, 30 windows of 24 hours over the last 150 days of the data.

---

In [ ]:
hh = res[res.dataset == "household"].reset_index(drop=True)
hh[[c for c in cols if c in hh]].style.format(
    {"mase": "{:.3f}", "skill": "{:+.1%}", "mae": "{:.3f}", "wape": "{:.1%}", "seconds": "{:.1f}"}, na_rep="").hide(axis="index")

model,family,mase,skill,mae,wape,seconds,notebook
"N-BEATS from scratch (PyTorch), 3-seed ensemble",deep learning,0.607,+24.3%,0.395,43.7%,8.4,05_Deep_Learning/05_NBEATS_From_Scratch
"MLP (PyTorch), 168 h lookback, 3-seed ensemble",deep learning,0.620,+22.8%,0.403,44.5%,1.8,05_Deep_Learning/01_DL_Data_Pipeline
"TiRex, zero-shot",foundation model,0.623,+23.6%,0.399,44.1%,4.9,06_Foundation_Models/02_Foundation_Model_Comparison
"MLP (PyTorch), instance norm, 168 h lookback",deep learning,0.629,+21.8%,0.409,45.1%,0.6,05_Deep_Learning/01_DL_Data_Pipeline
TFT (neuralforecast),deep learning,0.638,+20.4%,0.416,45.9%,152.5,05_Deep_Learning/04_Transformers
"Chronos-2, fine-tuned (300 steps)",foundation model,0.639,+22.6%,0.404,44.6%,93.8,06_Foundation_Models/03_Fine_Tuning
"DeepAR (neuralforecast, median)",deep learning,0.641,+20.3%,0.416,46.0%,75.1,05_Deep_Learning/02_RNN_LSTM_DeepAR
"Chronos-Bolt (small), zero-shot",foundation model,0.642,+20.3%,0.416,46.0%,0.3,06_Foundation_Models/02_Foundation_Model_Comparison
"Chronos-Bolt tiny, zero-shot (CPU)",foundation model,0.643,+19.8%,,,0.4,06_Foundation_Models/04_Small_Models_on_CPU
"Chronos-2, zero-shot",foundation model,0.644,+21.7%,0.409,45.2%,0.9,06_Foundation_Models/01_Chronos2_Zero_Shot


In [ ]:
order = hh.sort_values("skill")
show(bar_chart(order.model, {"skill vs seasonal naive": order.skill}, horizontal=True,
               height=f"{max(240, 40 * len(order) + 80)}px"))

## M4 Hourly

414 series, each scored on its fixed 48-hour test horizon, averaged across series.

---

In [ ]:
m4 = res[res.dataset == "m4_hourly"].reset_index(drop=True)
m4[[c for c in ["model", "family", "mase", "smape", "skill", "notebook"] if c in m4]].style.format(
    {"mase": "{:.3f}", "smape": "{:.2f}", "skill": "{:+.1%}"}, na_rep="").hide(axis="index")

model,family,mase,smape,skill,notebook
"LightGBM, global (diff 24 + scaler)",machine learning,0.887,11.68,,04_Machine_Learning/03_Global_vs_Local
"LightGBM, local per series (diff 24 + scaler)",machine learning,0.933,12.09,,04_Machine_Learning/03_Global_vs_Local
Seasonal naive (24 h),baseline,1.193,13.91,+0.0%,02_Evaluation/02_Benchmarks


In [ ]:
#| hide
assert {"household", "m4_hourly"} <= set(res.dataset)
assert (res.groupby("dataset").model.apply(lambda m: m.is_unique)).all(), "duplicate model names in one dataset"
assert hh.mase.is_monotonic_increasing